**Description** : Phase finale de prédiction et de recommandation.

**Objectifs** : Comparaison de modèles de classification (Random Forest vs Régression Logistique).

**Résultats** : Évaluation via Accuracy, F1-Score et Matrice de Confusion. Utilisation de la Régression Logistique pour générer des probabilités de futurs recrutements ("Faux Positifs" à fort potentiel).

***Cellule 1 : Préparation des données***

In [4]:
#importation les bibliothéques nécessaires 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [5]:
# Chargement du dataset crée au notebook 05
df_final = pd.read_csv("../resultats/dataset_prediction_liens.csv")

Ici , j'ai chargé la dataset que j'ai préparer pour la phase de machine learning

In [10]:
# Sélection des features (on exclut les IDs et la cible)
features_cols = [ 'comp_degree', 'preferential_attachment', 'cluster_id', 'same_location', 'same_industry']
X = df_final[features_cols]
y = df_final['target']

J'ai sélectionné seulement les features les plus importants ayant un impact sur ma variable cible

In [11]:
# Split 70% train / 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"✅ Données prêtes. Taille test : {len(X_test)} échantillons.")

✅ Données prêtes. Taille test : 600 échantillons.


J'ai divisé mon dataset en train data pour la phase d'entrainement et test data pour la phase d'évaluation

**Cellule 2 : Entraînement des modèles ML**

In [12]:
# 1. Logistic Regression (Modèle simple/linéaire)
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# 2. Random Forest (Modèle complexe/non-linéaire)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("✅ Entraînement des modèles ML terminé.")

✅ Entraînement des modèles ML terminé.


## 🔷 Algorithmes de classification utilisés

Dans ce projet de prédiction de liens, deux algorithmes de Machine Learning ont été utilisés : **la régression logistique** et **la forêt aléatoire (Random Forest)**.

---

# 🔹 1. Régression Logistique

### ✔️ Principe

La régression logistique est un algorithme de classification binaire qui estime la probabilité qu’un lien existe entre un employé et une entreprise.

---



---

### ✔️ Interprétation

* Le modèle calcule une probabilité entre 0 et 1
* Si la probabilité est supérieure à un seuil (ex : 0.5), le lien est prédit comme existant

---

### ✔️ Avantages

* Simple et rapide
* Facile à interpréter
* Fonctionne bien sur des relations linéaires

---

### ✔️ Inconvénients

* Ne capture pas les relations complexes
* Performance limitée sur des données non linéaires

---

# 🔹 2. Random Forest (Forêt Aléatoire)

### ✔️ Principe

La Random Forest est un algorithme basé sur un ensemble de plusieurs arbres de décision. Chaque arbre fait une prédiction, et le résultat final est obtenu par vote majoritaire.

---

### ✔️ Fonctionnement

* Construction de plusieurs arbres de décision
* Chaque arbre est entraîné sur un sous-échantillon des données
* Les prédictions sont combinées pour obtenir un résultat final

---

### ✔️ Interprétation

* Chaque arbre “vote” pour la présence ou non d’un lien
* La classe finale est celle qui obtient le plus de votes

---

### ✔️ Avantages

* Très performant sur des données complexes
* Capte les relations non linéaires
* Robuste au bruit et au surapprentissage

---

### ✔️ Inconvénients

* Moins interprétable qu’une régression logistique
* Plus coûteux en calcul

---




***Cellule 3 : Évaluation et choix du meilleur modèle***

In [13]:
def calculer_metriques(y_true, y_pred, nom_modele):
    return {
        "Modèle": nom_modele,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Précision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred)
    }

stats = [
    calculer_metriques(y_test, y_pred_lr, "Régression Logistique"),
    calculer_metriques(y_test, y_pred_rf, "Random Forest")
]

df_perf = pd.DataFrame(stats)
print(df_perf.to_string(index=False))

# Sauvegarde pour le rapport final
df_perf.to_csv("../resultats/performance_modeles.csv", index=False)

               Modèle  Accuracy  Précision   Recall  F1-Score
Régression Logistique  0.971667   0.989619 0.953333  0.971138
        Random Forest  0.970000   0.982877 0.956667  0.969595


## 🔷 Métriques d’évaluation du modèle

Pour évaluer la performance des modèles de prédiction de liens, plusieurs métriques de classification ont été utilisées.

---

# 🔹 1. Accuracy (Exactitude)

### ✔️ Définition

L’accuracy mesure la proportion de prédictions correctes parmi toutes les prédictions.

Accuracy = \frac{TP + TN}{TP + TN + FP + FN}

---

### ✔️ Interprétation

* Valeur proche de 1 : modèle performant
* Valeur faible : modèle peu fiable

👉 Attention : cette métrique peut être trompeuse si les classes sont déséquilibrées.

---

# 🔹 2. Précision (Precision)

### ✔️ Définition

La précision mesure la proportion de prédictions positives correctes.



---

### ✔️ Interprétation

* Haute précision :

  * peu de faux positifs
  * les liens prédits sont généralement corrects

👉 Utile lorsque le coût des fausses prédictions positives est élevé.

---

# 🔹 3. Recall (Rappel)

### ✔️ Définition

Le recall mesure la capacité du modèle à détecter tous les vrais liens.



---

### ✔️ Interprétation

* Recall élevé :

  * peu de faux négatifs
  * le modèle détecte la majorité des liens existants

👉 Important lorsque l’on veut éviter de manquer des liens potentiels.

---

# 🔹 4. F1-Score

### ✔️ Définition

Le F1-score est la moyenne harmonique entre la précision et le recall.



---

### ✔️ Interprétation

* Combine précision et recall
* Utile lorsque les classes sont déséquilibrées
* Permet un bon compromis entre faux positifs et faux négatifs

---

# 🔹 Conclusion

Ces métriques sont complémentaires :

* **Accuracy** : performance globale
* **Précision** : fiabilité des prédictions positives
* **Recall** : capacité à détecter les vrais liens
* **F1-score** : équilibre entre précision et recall

👉 Leur utilisation conjointe permet une évaluation complète du modèle de prédiction de liens.


## 🔷 Interprétation des résultats des modèles

Les résultats obtenus montrent que la **régression logistique** a fourni des performances plus pertinentes que le modèle Random Forest sur ce jeu de données.

---

### 🔹 Analyse

Ce résultat peut s’expliquer par plusieurs facteurs :

* Les relations entre les variables explicatives et la variable cible semblent être **principalement linéaires**
* Les features utilisées (degré, preferential attachment, similarité, etc.) sont relativement simples et bien structurées
* Le dataset ne présente pas une complexité suffisante pour exploiter pleinement la puissance d’un modèle plus complexe comme Random Forest

---

### 🔹 Interprétation

La régression logistique, en tant que modèle simple et interprétable, est donc mieux adaptée dans ce contexte.
Elle permet de capturer efficacement les relations entre les variables sans introduire de complexité inutile.




***Génération des recommandations***

Le but de ce modéle d'appliquer le modéle de régression logistique sur nos données , on filtre notre data lorsque le target vaut 0.et on va voir les probablité qu'un emloyé pourrait travailler dans une entreprise ou non 

In [14]:
# 1. On filtre pour ne garder QUE les cas où l'employé ne travaille pas dans l'entreprise (target == 0)
df_futur = df_final[df_final['target'] == 0].copy()

In [15]:
# 2. On isole les features à donner au modèle
X_futur = df_futur[features_cols]

In [ ]:
# 3. On calcule la probabilité de recrutement avec la Régression Logistique
# predict_proba renvoie la probabilité d'appartenir à la classe 1 (le lien se crée)
df_futur['probabilite_recrutement'] = lr_model.predict_proba(X_futur)[:, 1]

In [ ]:
# 4. On trie du plus probable au moins probable
df_recommandations = df_futur.sort_values(by='probabilite_recrutement', ascending=False)

In [18]:
# 5. Affichage du Top 10 des recrutements les plus probables
print("🌟 TOP 10 des prédictions de futurs recrutements :")
colonnes_a_afficher = ['employee_id', 'company_name', 'probabilite_recrutement']
print(df_recommandations[colonnes_a_afficher].head(10).to_string(index=False))

🌟 TOP 10 des prédictions de futurs recrutements :
                  employee_id                             company_name  probabilite_recrutement
       keith-freeman-379a1111                                      NaN                 1.000000
        hussam-alaa-09843411a                             at rich bake                 0.979878
  harrison-ainsworth-5b42b951                   blackberry corporation                 0.978791
             gail-fride-dwyer                            self employed                 0.289221
ronnie-caslin-caslin-73b835a2                            self employed                 0.289221
        manuela-dias-b868b29b                                  infosys                 0.164341
      carlton-frank-6bba271b3              national geographic society                 0.158117
           tony-diaz-a9599087 universit de fribourguniversitt freiburg                 0.158117
 iswarjono-iswarjono-5679941a                                   amazon                

In [19]:
# 6. Sauvegarde dans un fichier CSV dédié
chemin_reco = "../resultats/recommandations_futures_carrieres.csv"
df_recommandations[colonnes_a_afficher].to_csv(chemin_reco, index=False)

print(f"\n✅ Fichier de prédictions sauvegardé avec succès : {chemin_reco}")


✅ Fichier de prédictions sauvegardé avec succès : ../resultats/recommandations_futures_carrieres.csv
